# Original ApET: full MMStar baseline

Enable Internet and GPU T4 ×2 after pushing the update. This runs the original native model across 1,500 samples and four configurations with one model load. Native algorithms/dependencies are unchanged; our v2 scorer and native-source reference are recorded. Follow `research/ORIGINAL_MMSTAR_FULL.md`. Download current results before stopping a session.

## 1. Get code and choose result directory

In [ ]:
import json
import subprocess
from datetime import datetime, timezone
from pathlib import Path

REPO_URL = "https://github.com/doguineyes/CSE590-ApET.git"
REPO_REF = "apet-analysis"
repo = Path("/kaggle/working/CSE590-ApET")
if not repo.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)
subprocess.run(["git", "fetch", "origin"], cwd=repo, check=True)
subprocess.run(["git", "checkout", REPO_REF], cwd=repo, check=True)
subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True)
subprocess.run(["git", "rev-parse", "HEAD"], cwd=repo, check=True)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run_dir = Path(f"/kaggle/working/original-mmstar-1500-{stamp}")
run_dir.mkdir()
full_report = run_dir / "original-mmstar-1500.json"
original_python = repo / "environments/original/.venv/bin/python"
checkpoint_args = []  # Optional: ["--model", "/kaggle/input/...", "--vision-model", "/kaggle/input/..."]
print(f"Results: {run_dir}")


## 2. Environment, checkpoint storage, source snapshots

Cached weights are reused. A restarted session can download the pinned checkpoints within a 20-GiB budget; model weights stay outside Output.

In [ ]:
import shutil

gpus = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    capture_output=True, text=True, check=True,
).stdout.strip().splitlines()
if len(gpus) != 2 or any("T4" not in name for name in gpus):
    raise RuntimeError(f"Enable GPU T4 x2 before installing/running: {gpus}")
subprocess.run(
    ["bash", "bootstrap/setup_original_linux.sh", "--require-cuda", "--expected-gpus", "2",
     "--output", str(run_dir / "original-runtime.json")],
    cwd=repo, check=True,
)
subprocess.run(
    [str(original_python), "-I", "scripts/smoke_original_apet.py", "--preflight-only",
     "--download-budget-gib", "20", *checkpoint_args,
     "--output", str(run_dir / "original-preflight.json")],
    cwd=repo, check=True,
)
preflight = json.loads((run_dir / "original-preflight.json").read_text())
print(json.dumps(preflight["storage"], indent=2))
if not preflight["storage"]["fits"]:
    raise RuntimeError("Checkpoint storage/budget preflight failed; inspect the report before inference.")

reference = json.loads((repo / "research/provenance/original_llava_reference.json").read_text())
snapshot_files = [*reference["native_source_sha256"],
    "scripts/check_original_env.py", "scripts/smoke_original_apet.py",
    "scripts/eval_original_mmstar.py", "scripts/eval_mmstar.py",
    "scripts/mmstar_scoring.py", "scripts/rescore_original_mmstar.py",
    "bootstrap/setup_original_linux.sh", "environments/original/pyproject.toml",
    "environments/original/uv.lock", "environments/original/.python-version",
    "research/provenance/original_llava_reference.json", "research/IMPLEMENTATION_SCOPE.md",
]
for name in snapshot_files:
    destination = run_dir / "source" / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(repo / name, destination)


## 3. Full paired evaluation

6,000 predictions, seed 590, max 16 output tokens. Roughly one hour based on the pilot, excluding setup/downloads. If interrupted, completed rows are preserved but automatic resume is not implemented.

In [ ]:
subprocess.run(
    [str(original_python), "-I", "scripts/eval_original_mmstar.py", "--all-samples",
     "--configurations", "baseline", "input-only", "decoder-only", "combined",
     "--seed", "590", "--max-new-tokens", "16", "--download-budget-gib", "20",
     *checkpoint_args, "--output", str(full_report)],
    cwd=repo, check=True,
)


## 4. Inspect and download

Run this cell after a failure too, to preserve available evidence. The ZIP contains predictions, images, reports, and source/environment snapshots; model caches are excluded.

In [ ]:
import zipfile
from IPython.display import FileLink, display

if full_report.is_file():
    result = json.loads(full_report.read_text())
    print("Status:", result["status"], "Progress:", result.get("progress"))
    if result.get("error"):
        print("Error:", result["error"])
    for name, metrics in result.get("summaries", {}).items():
        apr = result.get("comparisons", {}).get(name, {}).get("apr")
        print(name, {key: metrics.get(key) for key in
                    ["evaluated", "correct", "accuracy", "unparsed", "strict_accuracy", "format_compliance_rate"]},
              "APR:", apr)
else:
    print("No inference report yet; bundling available setup/preflight evidence.")

bundle = run_dir.with_suffix(".zip")
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(run_dir.rglob("*")):
        if path.is_file():
            archive.write(path, str(path.relative_to(run_dir)))
print(f"Download before ending the session: {bundle}")
display(FileLink(str(bundle)))
